# FTIR spectral images — reproducible HOG + Linear SVM baseline

**Task:** classify original *folder labels* (`alcool`, `amina`, `cetona`), **not** verified multi-label chemistry.

**Reported prior run:** 306 training / 71 validation / 71 test images; validation accuracy **0.9155**, macro-F1 **0.9123**; test accuracy **0.8732**, macro-F1 **0.8709** (62/71 correct).

This notebook intentionally contains **no manufactured model outputs**. Re-run it to reproduce metrics; exact results may depend on package versions. The images are **not distributed in this repository**. Mount your own authorized copy under `MyDrive/espectros/{alcool,amina,cetona}`.

**Limitations:** folder-label errors; potentially equivalent compounds with different names; different sample preparations (e.g. KBr/Nujol); unverified spectral-image comparability. Results are **preliminary**, not validated functional-group identification.

## 1. Environment

In [ ]:
%pip -q install "pandas>=2,<3" "numpy>=1.26,<3" "Pillow>=10,<13" "scikit-image>=0.22,<0.27" "scikit-learn>=1.4,<1.9" joblib matplotlib

In [ ]:
import sys, numpy, pandas, sklearn, skimage, PIL
print({"python": sys.version.split()[0], "numpy": numpy.__version__,
       "pandas": pandas.__version__, "scikit_learn": sklearn.__version__,
       "scikit_image": skimage.__version__, "pillow": PIL.__version__})

## 2. Clone repository and mount authorized spectral images

In [ ]:
from pathlib import Path
import subprocess
repo = Path("/content/FTIR-Functional-Group-Classification")
if not (repo / "data/inventory_498.csv").exists():
    subprocess.run(["git", "clone",
                    "https://github.com/Arica80/FTIR-Functional-Group-Classification.git",
                    str(repo)], check=True)
from google.colab import drive
drive.mount("/content/drive")
base = Path("/content/drive/MyDrive/espectros")
inventory = __import__("pandas").read_csv(repo/"data/inventory_498.csv")
assert len(inventory) == 498, "Unexpected inventory size"
print(inventory["particao_historica"].value_counts())

## 3. Feature extraction — exact baseline parameters

In [ ]:
import numpy as np
from PIL import Image
from skimage.feature import hog
X, y, partitions, paths = [], [], [], []
for _, r in inventory.iterrows():
    part = str(r["particao_historica"])
    if part not in ("treino", "validacao", "teste"):
        continue
    image_path = base / str(r["familia_original"]) / str(r["arquivo"])
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    with Image.open(image_path) as img:
        gray = np.asarray(img.convert("L").resize((256, 256)))
    X.append(hog(gray, orientations=9, pixels_per_cell=(16,16),
                 cells_per_block=(2,2), block_norm="L2-Hys"))
    y.append(str(r["familia_original"]))
    partitions.append(part)
    paths.append(str(image_path))
X = np.asarray(X)
y = np.asarray(y)
partitions = np.asarray(partitions)
train = partitions == "treino"
validation = partitions == "validacao"
test = partitions == "teste"
assert (train.sum(), validation.sum(), test.sum()) == (306, 71, 71)
print("Feature matrix:", X.shape)

## 4. Fit SVM on training data only

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
model = make_pipeline(
    StandardScaler(),
    LinearSVC(C=0.1, class_weight="balanced", random_state=42, max_iter=10000)
)
model.fit(X[train], y[train])

## 5. Evaluate fixed historical splits (no retuning on test set)

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

out = Path("/content/ftir_results")
out.mkdir(exist_ok=True)
metrics = {}
for split_name, mask in [("validation", validation), ("test", test)]:
    prediction = model.predict(X[mask])
    metrics[split_name] = {
        "accuracy": float(accuracy_score(y[mask], prediction)),
        "macro_f1": float(f1_score(y[mask], prediction, average="macro")),
        "samples": int(mask.sum())
    }
    print("\n", split_name.upper(), metrics[split_name])
    print(classification_report(y[mask], prediction, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y[mask], prediction, labels=model.classes_,
        cmap="Blues", xticks_rotation=30)
    plt.title(f"HOG + SVM — {split_name}")
    plt.tight_layout()
    plt.savefig(out/f"confusion_matrix_{split_name}.png", dpi=200)
    plt.show()
(out/"metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

## 6. Export errors and the model

In [ ]:
import joblib
pred_test = model.predict(X[test])
test_rows = inventory.loc[
    inventory["particao_historica"].eq("teste")
].copy()
assert len(test_rows) == len(pred_test)
test_rows["predicted"] = pred_test
test_rows["correct"] = test_rows["familia_original"].eq(test_rows["predicted"])
errors = test_rows.loc[~test_rows["correct"]]
errors.to_csv(out/"test_errors.csv", index=False)
joblib.dump(model, out/"hog_svm_baseline.joblib")
print("Correct:", int(test_rows["correct"].sum()),
      "Errors:", len(errors))
print("Saved to:", out)
# Published results should match only after a real run on the authorized image set.